# Day 41 — Practice 2 & 3: Hive → Spark + Transformasi

**Tujuan:** Koneksi ke Hive External Table dan transformasi data menggunakan Spark.

Alur lengkap yang sudah dibangun:
```
PostgreSQL           HDFS Parquet          Hive                    Spark
(AdventureWorks) → (Data Lake)        → (Data Warehouse)  →  (Data Processing)
                    /datalake/raw/        adventureworks.           Aggregasi
                    sales/...             fact_sales_orders         Join
                    production/...        dim_product               Window Function
                    person/...            dim_customer              Curated Tables
```

**Yang dipraktikkan:**
1. Koneksi Spark ke Hive Metastore
2. Read Hive External Table
3. Transformasi: filter, join, groupBy, agregasi
4. Window functions
5. Simpan hasil transformasi ke Hive sebagai curated table

> **Prasyarat:** Jalankan dulu notebook Day 40 — `03_hdfs_to_hive.ipynb`

## 0. Persiapan: Buat Data Lake di HDFS

Bagian ini mengekstrak data dari PostgreSQL AdventureWorks ke HDFS sebagai Parquet,
lalu membuat Hive External Tables — sehingga notebook ini *self-contained*.

> Lewati bagian ini jika `02_adventureworks_to_hdfs.ipynb` dan `03_hdfs_to_hive.ipynb`
> sudah pernah dijalankan sebelumnya.

### 0.1 Setup SparkSession JDBC (tanpa Hive support)

In [ ]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F_jdbc
import subprocess

spark_jdbc = SparkSession.builder \
    .master('local[*]') \
    .appName('AdventureWorks-to-HDFS') \
    .config('spark.sql.parquet.compression.codec', 'snappy') \
    .getOrCreate()

JDBC_URL   = 'jdbc:postgresql://adventureworks-postgres:5432/postgres'
JDBC_PROPS = {'user':'postgres','password':'My_password1','driver':'org.postgresql.Driver'}
HDFS_BASE  = 'hdfs://namenode:9000/datalake/raw'

def read_pg(schema, table, query=None):
    src = query if query else f'"{schema}"."{table}"'
    return spark_jdbc.read.jdbc(url=JDBC_URL, table=src, properties=JDBC_PROPS)

print(f'Spark {spark_jdbc.version} ready (JDBC mode)')


### 0.2 Extract Tabel Dimensi ke HDFS

In [ ]:
dim_tables = [
    ('Production', 'Product'),
    ('Production', 'ProductCategory'),
    ('Production', 'ProductSubcategory'),
    ('Person',     'Person'),
    ('HumanResources', 'Department'),
    ('HumanResources', 'Employee'),
    ('Sales',      'Customer'),
    ('Sales',      'SalesTerritory'),
]
for schema, table in dim_tables:
    df = read_pg(schema, table)
    hdfs_path = f'{HDFS_BASE}/{schema}/{table}'
    df.write.mode('overwrite').parquet(hdfs_path)
    print(f'  {schema}.{table:30s} -> {df.count():6,} rows -> HDFS')
print('Semua tabel dimensi berhasil disimpan ke HDFS!')


### 0.3 Extract Tabel Fakta dengan Partisi

In [ ]:
# SalesOrderHeader - partisi order_year / order_month
df_orders = read_pg('Sales', 'SalesOrderHeader')
df_orders_partitioned = df_orders \
    .withColumn('order_year',  F_jdbc.year('orderdate')) \
    .withColumn('order_month', F_jdbc.month('orderdate'))
orders_path = f'{HDFS_BASE}/Sales/SalesOrderHeader'
df_orders_partitioned.write.mode('overwrite') \
    .partitionBy('order_year', 'order_month').parquet(orders_path)
print(f'  SalesOrderHeader -> {df_orders.count():,} rows -> HDFS (partisi order_year/order_month)')

# SalesOrderDetail - partisi order_year
df_detail = read_pg('Sales', 'SalesOrderDetail')
df_detail_year = df_detail \
    .join(df_orders.select('salesorderid', 'orderdate'), 'salesorderid') \
    .withColumn('order_year', F_jdbc.year('orderdate')) \
    .drop('orderdate')
detail_path = f'{HDFS_BASE}/Sales/SalesOrderDetail'
df_detail_year.write.mode('overwrite') \
    .partitionBy('order_year').parquet(detail_path)
print(f'  SalesOrderDetail  -> {df_detail.count():,} rows -> HDFS (partisi order_year)')

spark_jdbc.stop()
print('Data Lake selesai. spark_jdbc dihentikan.')


### 0.4 Buat Hive External Tables

Setelah data ada di HDFS, buat External Tables di Hive Metastore,
sekaligus inisialisasi SparkSession dengan Hive support yang dipakai sepanjang notebook.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Hive-Spark-Transformations') \
    .config('spark.jars', '/home/jovyan/work/jars/postgresql-42.7.3.jar') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

HDFS_BASE    = 'hdfs://namenode:9000/datalake/raw'
HDFS_CURATED = 'hdfs://namenode:9000/datalake/curated'

print(f'Spark {spark.version} dengan Hive support aktif')
spark.sql('SHOW DATABASES').show()


In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS adventureworks COMMENT 'AdventureWorks Data Warehouse'")
spark.sql('USE adventureworks')
print('Database adventureworks siap')
spark.sql('SHOW DATABASES').show()


In [ ]:
def create_external_table(table_name, hdfs_path):
    spark.sql(f'DROP TABLE IF EXISTS adventureworks.{table_name}')
    spark.sql(f"""
        CREATE EXTERNAL TABLE adventureworks.{table_name}
        USING PARQUET
        LOCATION '{hdfs_path}'
    """)
    count = spark.sql(f'SELECT COUNT(*) FROM adventureworks.{table_name}').collect()[0][0]
    print(f'  {table_name:35s} -> {count:6,} rows')

dim_tables_hive = [
    ('dim_product',          f'{HDFS_BASE}/Production/Product'),
    ('dim_product_category', f'{HDFS_BASE}/Production/ProductCategory'),
    ('dim_product_subcat',   f'{HDFS_BASE}/Production/ProductSubcategory'),
    ('dim_person',           f'{HDFS_BASE}/Person/Person'),
    ('dim_employee',         f'{HDFS_BASE}/HumanResources/Employee'),
    ('dim_department',       f'{HDFS_BASE}/HumanResources/Department'),
    ('dim_customer',         f'{HDFS_BASE}/Sales/Customer'),
    ('dim_territory',        f'{HDFS_BASE}/Sales/SalesTerritory'),
]
for tname, path in dim_tables_hive:
    create_external_table(tname, path)
print('Semua tabel dimensi berhasil dibuat di Hive!')


In [ ]:
orders_path = f'{HDFS_BASE}/Sales/SalesOrderHeader'
detail_path  = f'{HDFS_BASE}/Sales/SalesOrderDetail'

# fact_sales_orders - partisi order_year / order_month
spark.sql('DROP TABLE IF EXISTS adventureworks.fact_sales_orders')
spark.sql(f"""
    CREATE EXTERNAL TABLE adventureworks.fact_sales_orders (
        SalesOrderID           INT,
        RevisionNumber         SMALLINT,
        OrderDate              TIMESTAMP,
        DueDate                TIMESTAMP,
        ShipDate               TIMESTAMP,
        Status                 SMALLINT,
        OnlineOrderFlag        BOOLEAN,
        SalesOrderNumber       STRING,
        PurchaseOrderNumber    STRING,
        AccountNumber          STRING,
        CustomerID             INT,
        SalesPersonID          INT,
        TerritoryID            INT,
        BillToAddressID        INT,
        ShipToAddressID        INT,
        ShipMethodID           INT,
        CreditCardID           INT,
        CreditCardApprovalCode STRING,
        CurrencyRateID         INT,
        SubTotal               DECIMAL(19,4),
        TaxAmt                 DECIMAL(19,4),
        Freight                DECIMAL(19,4),
        TotalDue               DECIMAL(19,4),
        Comment                STRING,
        rowguid                STRING,
        ModifiedDate           TIMESTAMP
    )
    PARTITIONED BY (order_year INT, order_month INT)
    STORED AS PARQUET
    LOCATION '{orders_path}'
""")
spark.sql('MSCK REPAIR TABLE adventureworks.fact_sales_orders')
cnt = spark.sql('SELECT COUNT(*) FROM adventureworks.fact_sales_orders').collect()[0][0]
print(f'  fact_sales_orders  -> {cnt:,} rows')

# fact_order_details - partisi order_year
spark.sql('DROP TABLE IF EXISTS adventureworks.fact_order_details')
spark.sql(f"""
    CREATE EXTERNAL TABLE adventureworks.fact_order_details (
        SalesOrderID          INT,
        SalesOrderDetailID    INT,
        CarrierTrackingNumber STRING,
        OrderQty              SMALLINT,
        ProductID             INT,
        SpecialOfferID        INT,
        UnitPrice             DECIMAL(19,4),
        UnitPriceDiscount     DECIMAL(19,4),
        LineTotal             DECIMAL(38,6),
        rowguid               STRING,
        ModifiedDate          TIMESTAMP
    )
    PARTITIONED BY (order_year INT)
    STORED AS PARQUET
    LOCATION '{detail_path}'
""")
spark.sql('MSCK REPAIR TABLE adventureworks.fact_order_details')
cnt2 = spark.sql('SELECT COUNT(*) FROM adventureworks.fact_order_details').collect()[0][0]
print(f'  fact_order_details -> {cnt2:,} rows')

print()
spark.sql('SHOW TABLES IN adventureworks').show(30)


## 1. Baca Tabel dari Hive External Table

In [ ]:
# Set default database
spark.sql('USE adventureworks')

print('=== Tabel yang tersedia di Hive ===')
spark.sql('SHOW TABLES').show(30)

# --- Load semua tabel dari Hive ---
# Cara 1: via spark.table() — langsung dari Hive Metastore
fact_orders  = spark.table('adventureworks.fact_sales_orders')
fact_details = spark.table('adventureworks.fact_order_details')
dim_product  = spark.table('adventureworks.dim_product')
dim_product_cat = spark.table('adventureworks.dim_product_category')
dim_customer = spark.table('adventureworks.dim_customer')
dim_person   = spark.table('adventureworks.dim_person')
dim_employee = spark.table('adventureworks.dim_employee')
dim_dept     = spark.table('adventureworks.dim_department')

# Cara 2: via spark.sql() — lebih SQL-like
# fact_orders = spark.sql('SELECT * FROM adventureworks.fact_sales_orders')

print('Semua tabel berhasil di-load dari Hive!')
print(f'fact_orders  : {fact_orders.count():,} rows')
print(f'fact_details : {fact_details.count():,} rows')
print(f'dim_product  : {dim_product.count():,} rows')

## 2. Transformasi Dasar: Filter & Select

In [ ]:
# Filter: ambil data tahun 2013 ke atas
recent_orders = fact_orders.filter(F.col('order_year') >= 2013)

print(f'Orders >= 2013: {recent_orders.count():,} rows')

# Select kolom yang relevan saja
orders_clean = fact_orders.select(
    'salesorderid',
    'orderdate',
    'customerid',
    'salespersonid',
    'subtotal',
    'taxamt',
    'freight',
    'totaldue',
    'order_year',
    'order_month'
)

orders_clean.show(3)

## 3. Transformasi Join: Sales dengan Produk dan Customer

In [ ]:
# Join fact_details + dim_product + fact_orders
# Hasilnya: setiap baris = 1 item order dengan informasi produk & order

df_sales_enriched = fact_details \
    .join(dim_product.select('productid', 'name', 'listprice', 'color', 'productline', 'productsubcategoryid'),
          on='productid', how='inner') \
    .join(fact_orders.select('salesorderid', 'orderdate', 'customerid', 'order_year', 'order_month', 'totaldue'),
          on='salesorderid', how='inner') \
    .select(
        'salesorderid', 'orderdate', 'customerid',
        'productid',
        F.col('name').alias('product_name'),
        F.col('color').alias('product_color'),
        F.col('productline').alias('product_line'),
        'orderqty', 'unitprice', 'linetotal',
        'order_year', 'order_month'
    )

print(f'Rows setelah join: {df_sales_enriched.count():,}')
df_sales_enriched.show(5, truncate=False)

## 4. Agregasi: Revenue Analysis

In [ ]:
# --- Revenue per tahun per product line ---
revenue_by_line = df_sales_enriched \
    .groupBy('order_year', 'product_line') \
    .agg(
        F.count('salesorderid').alias('total_orders'),
        F.sum('orderqty').alias('total_qty'),
        F.round(F.sum('linetotal'), 2).alias('total_revenue'),
        F.round(F.avg('unitprice'), 2).alias('avg_unit_price')
    ) \
    .orderBy('order_year', 'product_line')

print('=== Revenue per Tahun per Product Line ===')
revenue_by_line.show(20)

In [ ]:
# --- Top 10 produk terlaris per tahun ---
top_products = df_sales_enriched \
    .groupBy('order_year', 'productid', 'product_name', 'product_line') \
    .agg(
        F.sum('orderqty').alias('total_qty'),
        F.round(F.sum('linetotal'), 2).alias('total_revenue')
    ) \
    .orderBy('order_year', F.desc('total_revenue'))

print('=== Top 5 Produk per Tahun ===')
top_products.show(20)

## 5. Window Functions

Window functions memungkinkan kita menghitung nilai relatif terhadap kelompok baris
tanpa mengurangi jumlah baris (berbeda dengan groupBy).

In [ ]:
# --- 5.1 Rank produk berdasarkan revenue di tiap tahun ---
window_year = Window.partitionBy('order_year').orderBy(F.desc('total_revenue'))

product_revenue = df_sales_enriched \
    .groupBy('order_year', 'product_name') \
    .agg(F.round(F.sum('linetotal'), 2).alias('total_revenue'))

product_ranked = product_revenue \
    .withColumn('rank',       F.rank().over(window_year)) \
    .withColumn('dense_rank', F.dense_rank().over(window_year)) \
    .withColumn('row_number', F.row_number().over(window_year)) \
    .filter(F.col('rank') <= 3) \
    .orderBy('order_year', 'rank')

print('=== Top 3 Produk per Tahun (Window Rank) ===')
product_ranked.show(20, truncate=False)

In [ ]:
# --- 5.2 Running total revenue per bulan dalam satu tahun ---
window_cumulative = Window.partitionBy('order_year') \
                          .orderBy('order_month') \
                          .rowsBetween(Window.unboundedPreceding, Window.currentRow)

monthly_cumulative = fact_orders \
    .groupBy('order_year', 'order_month') \
    .agg(F.round(F.sum('totaldue'), 2).alias('monthly_revenue')) \
    .withColumn('cumulative_revenue', F.round(F.sum('monthly_revenue').over(window_cumulative), 2)) \
    .withColumn('pct_of_yearly', F.round(
        F.col('monthly_revenue') / F.sum('monthly_revenue').over(Window.partitionBy('order_year')) * 100, 1
    )) \
    .orderBy('order_year', 'order_month')

print('=== Cumulative Revenue per Bulan (2014) ===')
monthly_cumulative.filter(F.col('order_year') == 2014).show()

In [ ]:
# --- 5.3 Lag/Lead: perbandingan revenue bulan ini vs bulan sebelumnya ---
window_lag = Window.partitionBy('order_year').orderBy('order_month')

monthly_comparison = fact_orders \
    .groupBy('order_year', 'order_month') \
    .agg(F.round(F.sum('totaldue'), 2).alias('revenue')) \
    .withColumn('prev_month_revenue', F.lag('revenue', 1).over(window_lag)) \
    .withColumn('mom_growth_pct', F.round(
        (F.col('revenue') - F.col('prev_month_revenue')) / F.col('prev_month_revenue') * 100, 1
    )) \
    .orderBy('order_year', 'order_month')

print('=== Month-over-Month Growth (2014) ===')
monthly_comparison.filter(F.col('order_year') == 2014).show()

## 6. Simpan Hasil ke Hive: Curated / Gold Layer

Hasil transformasi disimpan kembali ke Hive sebagai **curated tables** — data siap konsumsi untuk analitik.

In [ ]:
# Buat schema curated di Hive
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks_curated COMMENT "Curated/Gold layer"')
spark.sql('USE adventureworks_curated')

print('Database adventureworks_curated siap')

In [ ]:
# --- Curated Table 1: Sales Summary per Tahun & Bulan ---
monthly_summary = fact_orders \
    .groupBy('order_year', 'order_month') \
    .agg(
        F.count('salesorderid').alias('total_orders'),
        F.countDistinct('customerid').alias('unique_customers'),
        F.round(F.sum('subtotal'), 2).alias('total_subtotal'),
        F.round(F.sum('totaldue'), 2).alias('total_revenue'),
        F.round(F.avg('totaldue'), 2).alias('avg_order_value'),
        F.round(F.max('totaldue'), 2).alias('max_order_value')
    )

# Simpan ke Hive (Managed Table — disimpan di warehouse)
monthly_summary.write \
    .mode('overwrite') \
    .partitionBy('order_year') \
    .saveAsTable('adventureworks_curated.monthly_sales_summary')

print('✓ monthly_sales_summary tersimpan')

In [ ]:
# --- Curated Table 2: Product Performance ---
product_performance = df_sales_enriched \
    .groupBy('order_year', 'productid', 'product_name', 'product_line', 'product_color') \
    .agg(
        F.count('salesorderid').alias('total_orders'),
        F.sum('orderqty').alias('total_qty_sold'),
        F.round(F.sum('linetotal'), 2).alias('total_revenue'),
        F.round(F.avg('unitprice'), 2).alias('avg_selling_price')
    )

product_performance.write \
    .mode('overwrite') \
    .partitionBy('order_year') \
    .saveAsTable('adventureworks_curated.product_performance')

print('✓ product_performance tersimpan')

In [ ]:
# --- Curated Table 3: Customer Profile ---
# Gabungkan data customer dengan historis pembelian
customer_profile = fact_orders \
    .groupBy('customerid') \
    .agg(
        F.count('salesorderid').alias('total_orders'),
        F.round(F.sum('totaldue'), 2).alias('lifetime_value'),
        F.round(F.avg('totaldue'), 2).alias('avg_order_value'),
        F.min('orderdate').alias('first_order_date'),
        F.max('orderdate').alias('last_order_date')
    ) \
    .join(dim_customer.select('customerid', 'personid', 'territoryid'), on='customerid', how='left') \
    .join(dim_person.select(
            F.col('businessentityid').alias('personid'),
            F.concat_ws(' ', 'firstname', 'lastname').alias('customer_name'),
            'emailpromotion'
         ), on='personid', how='left')

customer_profile.write \
    .mode('overwrite') \
    .saveAsTable('adventureworks_curated.customer_profile')

print('✓ customer_profile tersimpan')
print('\n=== Verifikasi ===')
spark.sql('SHOW TABLES IN adventureworks_curated').show()

## 7. Query Final: End-to-End via Hive

In [ ]:
# Query dari curated tables — data sudah bersih dan siap dipakai
print('=== Top 10 Customers by Lifetime Value ===')
spark.sql("""
    SELECT
        customerid,
        COALESCE(customer_name, 'Unknown') AS customer_name,
        total_orders,
        lifetime_value,
        avg_order_value,
        DATEDIFF(last_order_date, first_order_date) AS days_as_customer
    FROM adventureworks_curated.customer_profile
    ORDER BY lifetime_value DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
print('=== Monthly Sales Summary 2014 ===')
spark.sql("""
    SELECT
        order_month,
        total_orders,
        unique_customers,
        total_revenue,
        avg_order_value
    FROM adventureworks_curated.monthly_sales_summary
    WHERE order_year = 2014
    ORDER BY order_month
""").show()

## 8. Arsitektur Final yang Sudah Dibangun

```
┌─────────────────────────────────────────────────────────────────────┐
│  SOURCE                                                             │
│  PostgreSQL: chriseaton/adventureworks:postgres (port 5433)        │
└──────────────────────────┬──────────────────────────────────────────┘
                           │ Spark JDBC Read
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  DATA LAKE (HDFS)                                                   │
│  hdfs://namenode:9000/datalake/raw/                                 │
│  ├── sales/salesorderheader/     (partisi: order_year/order_month) │
│  ├── sales/salesorderdetail/     (partisi: order_year)             │
│  ├── production/product/                                            │
│  ├── person/person/                                                 │
│  └── humanresources/...                                             │
└──────────────────────────┬──────────────────────────────────────────┘
                           │ Hive External Table + MSCK REPAIR
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  DATA WAREHOUSE (HIVE)                                              │
│  adventureworks database:                                           │
│  ├── fact_sales_orders   (External, partitioned)                   │
│  ├── fact_order_details  (External, partitioned)                   │
│  ├── dim_product                                                    │
│  ├── dim_customer                                                   │
│  └── dim_person                                                     │
│                                                                     │
│  adventureworks_curated database:                                   │
│  ├── monthly_sales_summary   (Managed)                             │
│  ├── product_performance     (Managed)                             │
│  └── customer_profile        (Managed)                             │
└──────────────────────────┬──────────────────────────────────────────┘
                           │ Spark SQL / DBeaver
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  ANALYTICS / BI                                                     │
│  • DBeaver → HiveServer2 (port 10000)                              │
│  • Jupyter Notebook → Spark SQL                                     │
└─────────────────────────────────────────────────────────────────────┘
```

## Summary

Yang sudah dipraktikkan:
- ✅ `spark.table()` — read Hive External Table ke DataFrame
- ✅ Filter, Select, Join antar tabel Hive
- ✅ GroupBy dan agregasi (count, sum, avg, max)
- ✅ **Window functions**: `rank()`, `dense_rank()`, `row_number()`, `sum() over()`, `lag()`
- ✅ Simpan hasil ke Hive sebagai curated Managed Table
- ✅ End-to-end pipeline: PostgreSQL → HDFS → Hive → Spark → Curated Hive

🎉 **Stack lengkap sudah berdiri:**
- **Hadoop HDFS** = Data Lake (raw storage)
- **Hive** = Data Warehouse (schema, governance, SQL interface)
- **Spark** = Data Processing (transformasi, analytics, ML)